# 🔧 CARBON EMISSION DATA - PREPROCESSING & FEATURE ENGINEERING

**Project**: Carbon Emission Prediction Platform  
**Notebook**: 02 - Data Preprocessing  
**Purpose**: Clean data and prepare features for modeling

---

## 📋 What This Notebook Does

1. ✅ Load raw data
2. ✅ Handle missing values (if any)
3. ✅ **Time-Series Specific Outlier Detection & Treatment**
4. ✅ Feature Engineering (lag features, rolling statistics)
5. ✅ Stationarity Testing
6. ✅ Train/Test Split (time-based)
7. ✅ Save processed data for modeling

---

## ⚠️ IMPORTANT: Why We Use Special Methods for Time Series

**Your friends used Winsorizer (IQR method) - THIS IS WRONG for time series!**

We'll use:
- ✅ Seasonal Decomposition
- ✅ Rolling Z-Score Method
- ✅ Per-Series Treatment (not pooled)
- ✅ Interpolation (not capping)

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


---
## 1️⃣ LOAD RAW DATA

In [2]:
# Load the dataset
data = pd.read_csv('../data/raw/emissions.csv')

print("=" * 70)
print("DATA LOADED")
print("=" * 70)
print(f"Total Records: {len(data):,}")
print(f"Time Range: {data['year'].min()} - {data['year'].max()}")
print(f"States: {data['state-name'].nunique()}")
print(f"Sectors: {data['sector-name'].nunique()}")
print(f"Fuel Types: {data['fuel-name'].nunique()}")

# Create a copy for preprocessing
df = data.copy()
print("\n✅ Working copy created")

DATA LOADED
Total Records: 59,901
Time Range: 1970 - 2021
States: 52
Sectors: 6
Fuel Types: 4

✅ Working copy created


---
## 2️⃣ HANDLE MISSING VALUES

In [3]:
# Check for missing values
missing = df.isnull().sum()
print("Missing Values:")
print(missing)

if missing.sum() == 0:
    print("\n✅ No missing values found!")
else:
    print(f"\n⚠️  Found {missing.sum()} missing values")
    # Handle missing values in 'value' column using forward fill within groups
    df['value'] = df.groupby(['state-name', 'sector-name', 'fuel-name'])['value'].ffill()
    print("✅ Missing values handled using forward fill within time series groups")

Missing Values:
year           0
state-name     0
sector-name    0
fuel-name      0
value          0
dtype: int64

✅ No missing values found!


---
## 3️⃣ TIME-SERIES SPECIFIC OUTLIER DETECTION

### 🎯 Our Approach (CORRECT for Time Series):

1. **Separate each time series** (state-sector-fuel combination)
2. **Seasonal Decomposition** to separate trend/seasonal/residual
3. **Detect outliers in residuals only** (not raw values)
4. **Use 4-sigma threshold** (conservative)
5. **Interpolate extreme outliers** (don't cap them)
6. **Document everything**

In [4]:
def detect_and_handle_outliers(ts, method='conservative', min_periods=10):
    """
    Time-series specific outlier detection and handling.
    
    Parameters:
    -----------
    ts : pd.Series
        Time series data (sorted by time)
    method : str
        'conservative' - 4-sigma threshold (recommended)
        'moderate' - 3-sigma threshold
    min_periods : int
        Minimum number of observations required
    
    Returns:
    --------
    ts_clean : pd.Series
        Cleaned time series
    n_outliers : int
        Number of outliers detected
    outlier_indices : list
        Indices of outlier points
    """
    
    # Skip if series too short
    if len(ts) < min_periods:
        return ts, 0, []
    
    # Skip if constant series
    if ts.std() == 0 or ts.nunique() <= 2:
        return ts, 0, []
    
    ts_clean = ts.copy()
    outlier_indices = []
    
    try:
        # Method 1: Seasonal Decomposition (if enough data)
        if len(ts) >= 20:
            try:
                # Use period of 10 years or half the data length
                period = min(10, len(ts) // 2)
                decomposition = seasonal_decompose(ts, model='additive', period=period, extrapolate_trend='freq')
                
                residuals = decomposition.resid.dropna()
                
                # Set threshold based on method
                if method == 'conservative':
                    threshold = 4 * residuals.std()
                else:
                    threshold = 3 * residuals.std()
                
                # Detect outliers in residuals
                outliers = np.abs(residuals) > threshold
                outlier_indices = residuals[outliers].index.tolist()
                
            except:
                # If decomposition fails, use rolling method
                pass
        
        # Method 2: Rolling Z-Score (backup or for shorter series)
        if len(outlier_indices) == 0 and len(ts) >= min_periods:
            window = min(5, len(ts) // 3)
            rolling_mean = ts.rolling(window=window, center=True, min_periods=1).mean()
            rolling_std = ts.rolling(window=window, center=True, min_periods=1).std()
            
            # Avoid division by zero
            rolling_std = rolling_std.replace(0, ts.std())
            
            z_scores = np.abs((ts - rolling_mean) / rolling_std)
            
            # Set threshold
            if method == 'conservative':
                threshold = 4
            else:
                threshold = 3
            
            outliers = z_scores > threshold
            outlier_indices = ts[outliers].index.tolist()
        
        # Handle outliers using interpolation
        if len(outlier_indices) > 0:
            ts_clean.loc[outlier_indices] = np.nan
            ts_clean = ts_clean.interpolate(method='linear', limit_direction='both')
        
        return ts_clean, len(outlier_indices), outlier_indices
        
    except Exception as e:
        # If anything fails, return original series
        return ts, 0, []

print("✅ Outlier detection function defined")

✅ Outlier detection function defined


In [5]:
# Apply outlier detection to each time series
print("=" * 70)
print("APPLYING TIME-SERIES OUTLIER DETECTION")
print("=" * 70)
print("\nThis may take a few minutes...\n")

# Get unique combinations
unique_series = df.groupby(['state-name', 'sector-name', 'fuel-name'])

cleaned_groups = []
outlier_report = []

# Process each time series
for (state, sector, fuel), group in tqdm(unique_series, desc="Processing series"):
    # Sort by year
    group = group.sort_values('year').reset_index(drop=True)
    
    # Extract time series
    ts = group['value']
    
    # Detect and handle outliers
    ts_clean, n_outliers, outlier_idx = detect_and_handle_outliers(ts, method='conservative')
    
    # Update the group with cleaned values
    group_clean = group.copy()
    group_clean['value'] = ts_clean
    group_clean['n_outliers_removed'] = n_outliers
    
    cleaned_groups.append(group_clean)
    
    # Record outlier statistics
    if n_outliers > 0:
        outlier_report.append({
            'state': state,
            'sector': sector,
            'fuel': fuel,
            'n_outliers': n_outliers,
            'pct_outliers': round(100 * n_outliers / len(ts), 2),
            'original_mean': round(ts.mean(), 2),
            'cleaned_mean': round(ts_clean.mean(), 2)
        })

# Combine all cleaned groups
df_clean = pd.concat(cleaned_groups, ignore_index=True)

print("\n✅ Outlier detection complete!")
print(f"\nTotal series processed: {len(unique_series)}")
print(f"Series with outliers: {len(outlier_report)}")
print(f"Total outliers detected: {df_clean['n_outliers_removed'].sum()}")

APPLYING TIME-SERIES OUTLIER DETECTION

This may take a few minutes...



Processing series: 100%|██████████| 1236/1236 [00:01<00:00, 1089.45it/s]



✅ Outlier detection complete!

Total series processed: 1236
Series with outliers: 113
Total outliers detected: 5711


In [6]:
# Display outlier report
if len(outlier_report) > 0:
    outlier_df = pd.DataFrame(outlier_report).sort_values('n_outliers', ascending=False)
    print("\n📊 TOP 10 SERIES WITH MOST OUTLIERS:")
    print(outlier_df.head(10))
    
    # Save full report
    outlier_df.to_csv('../reports/outlier_analysis/outlier_report.csv', index=False)
    print("\n✅ Saved: reports/outlier_analysis/outlier_report.csv")
else:
    print("\n✅ No outliers detected using conservative method!")


📊 TOP 10 SERIES WITH MOST OUTLIERS:
              state                                           sector  \
2            Alaska          Transportation carbon dioxide emissions   
107         Vermont          Electric Power carbon dioxide emissions   
91     Pennsylvania          Electric Power carbon dioxide emissions   
0           Alabama          Electric Power carbon dioxide emissions   
71    New Hampshire          Transportation carbon dioxide emissions   
82   North Carolina             Residential carbon dioxide emissions   
81   North Carolina              Industrial carbon dioxide emissions   
80         New York          Transportation carbon dioxide emissions   
79         New York          Transportation carbon dioxide emissions   
78         New York  Total carbon dioxide emissions from all sectors   

            fuel  n_outliers  pct_outliers  original_mean  cleaned_mean  
2    Natural Gas           2          3.85           0.16          0.14  
107    Petroleum      

In [7]:
# Visualize before/after for a sample series
if len(outlier_report) > 0:
    # Get the series with most outliers
    top_outlier = outlier_df.iloc[0]
    
    # Get original and cleaned data
    original = df[
        (df['state-name'] == top_outlier['state']) &
        (df['sector-name'] == top_outlier['sector']) &
        (df['fuel-name'] == top_outlier['fuel'])
    ].sort_values('year')
    
    cleaned = df_clean[
        (df_clean['state-name'] == top_outlier['state']) &
        (df_clean['sector-name'] == top_outlier['sector']) &
        (df_clean['fuel-name'] == top_outlier['fuel'])
    ].sort_values('year')
    
    # Create comparison plot
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=original['year'],
        y=original['value'],
        mode='lines+markers',
        name='Original',
        line=dict(color='red', width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=cleaned['year'],
        y=cleaned['value'],
        mode='lines+markers',
        name='Cleaned',
        line=dict(color='green', width=2)
    ))
    
    fig.update_layout(
        title=f"Before/After Outlier Handling<br>{top_outlier['state']} - {top_outlier['sector']} - {top_outlier['fuel']}",
        xaxis_title='Year',
        yaxis_title='CO₂ Emissions',
        height=500,
        template='plotly_white'
    )
    
    # fig.show()  # Commented to avoid nbformat error
    fig.write_html('../reports/outlier_analysis/before_after_example.html')
    print("✅ Saved: reports/outlier_analysis/before_after_example.html")

✅ Saved: reports/outlier_analysis/before_after_example.html


---
## 4️⃣ FEATURE ENGINEERING

Create features for modeling:
- Lag features (previous years)
- Rolling statistics (moving averages)
- Year-based features

In [8]:
print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

# We'll create features for each time series group
feature_groups = []

for (state, sector, fuel), group in tqdm(df_clean.groupby(['state-name', 'sector-name', 'fuel-name']), 
                                          desc="Creating features"):
    group = group.sort_values('year').reset_index(drop=True)
    
    # Lag features (previous 1, 2, 3 years)
    group['lag_1'] = group['value'].shift(1)
    group['lag_2'] = group['value'].shift(2)
    group['lag_3'] = group['value'].shift(3)
    
    # Rolling statistics (3-year window)
    group['rolling_mean_3'] = group['value'].rolling(window=3, min_periods=1).mean()
    group['rolling_std_3'] = group['value'].rolling(window=3, min_periods=1).std()
    
    # Rolling statistics (5-year window)
    group['rolling_mean_5'] = group['value'].rolling(window=5, min_periods=1).mean()
    
    # Year-over-year change
    group['yoy_change'] = group['value'].pct_change()
    
    # Time-based features
    group['year_numeric'] = group['year'] - group['year'].min()  # Years since start
    group['decade'] = (group['year'] // 10) * 10
    
    feature_groups.append(group)

df_features = pd.concat(feature_groups, ignore_index=True)

print("\n✅ Feature engineering complete!")
print(f"\nNew columns created: {len(df_features.columns) - len(df_clean.columns)}")
print("\nNew features:")
new_cols = [col for col in df_features.columns if col not in df_clean.columns]
for col in new_cols:
    print(f"  - {col}")

FEATURE ENGINEERING


Creating features: 100%|██████████| 1236/1236 [00:01<00:00, 889.54it/s]



✅ Feature engineering complete!

New columns created: 9

New features:
  - lag_1
  - lag_2
  - lag_3
  - rolling_mean_3
  - rolling_std_3
  - rolling_mean_5
  - yoy_change
  - year_numeric
  - decade


In [9]:
# Display sample with features
print("\n📊 SAMPLE DATA WITH FEATURES:")
sample = df_features[
    (df_features['state-name'] == 'California') &
    (df_features['sector-name'] == 'Total carbon dioxide emissions from all sectors') &
    (df_features['fuel-name'] == 'All Fuels')
].tail(10)

print(sample[['year', 'value', 'lag_1', 'lag_2', 'rolling_mean_3', 'rolling_mean_5', 'yoy_change']])


📊 SAMPLE DATA WITH FEATURES:
      year       value       lag_1       lag_2  rolling_mean_3  \
5400  2012  348.750203  342.655381  356.592180      349.332588   
5401  2013  349.714192  348.750203  342.655381      347.039925   
5402  2014  345.386127  349.714192  348.750203      347.950174   
5403  2015  351.421508  345.386127  349.714192      348.840609   
5404  2016  353.372145  351.421508  345.386127      350.059927   
5405  2017  356.532043  353.372145  351.421508      353.775232   
5406  2018  358.605130  356.532043  353.372145      356.169773   
5407  2019  358.266355  358.605130  356.532043      357.801176   
5408  2020  303.815453  358.266355  358.605130      340.228979   
5409  2021  324.039053  303.815453  358.266355      328.706954   

      rolling_mean_5  yoy_change  
5400      360.471196    0.017787  
5401      353.613943    0.002764  
5402      348.619617   -0.012376  
5403      347.585482    0.017474  
5404      349.728835    0.005551  
5405      351.285203    0.008942 

---
## 5️⃣ STATIONARITY TESTING

Test if time series are stationary (important for ARIMA modeling)

In [10]:
def test_stationarity(ts, significance=0.05):
    """
    Test stationarity using ADF and KPSS tests.
    
    Returns:
    --------
    is_stationary : bool
    adf_pvalue : float
    kpss_pvalue : float
    """
    try:
        # Skip if too short or constant
        if len(ts) < 10 or ts.std() == 0:
            return None, None, None
        
        # ADF Test (H0: non-stationary)
        adf_result = adfuller(ts.dropna())
        adf_pvalue = adf_result[1]
        
        # KPSS Test (H0: stationary)
        kpss_result = kpss(ts.dropna(), nlags='auto')
        kpss_pvalue = kpss_result[1]
        
        # Series is stationary if:
        # - ADF p-value < 0.05 (reject H0, series is stationary)
        # - KPSS p-value > 0.05 (fail to reject H0, series is stationary)
        is_stationary = (adf_pvalue < significance) and (kpss_pvalue > significance)
        
        return is_stationary, adf_pvalue, kpss_pvalue
    
    except:
        return None, None, None

print("✅ Stationarity test function defined")

✅ Stationarity test function defined


In [11]:
# Test stationarity for each series
print("=" * 70)
print("TESTING STATIONARITY")
print("=" * 70)
print("\nThis may take a few minutes...\n")

stationarity_results = []

for (state, sector, fuel), group in tqdm(df_features.groupby(['state-name', 'sector-name', 'fuel-name']),
                                          desc="Testing stationarity"):
    ts = group.sort_values('year')['value'].dropna()
    
    is_stationary, adf_p, kpss_p = test_stationarity(ts)
    
    if is_stationary is not None:
        stationarity_results.append({
            'state': state,
            'sector': sector,
            'fuel': fuel,
            'is_stationary': is_stationary,
            'adf_pvalue': round(adf_p, 4),
            'kpss_pvalue': round(kpss_p, 4),
            'n_observations': len(ts)
        })

stationarity_df = pd.DataFrame(stationarity_results)

print("\n✅ Stationarity testing complete!")
print(f"\nTotal series tested: {len(stationarity_df)}")
print(f"Stationary series: {stationarity_df['is_stationary'].sum()} ({100*stationarity_df['is_stationary'].mean():.1f}%)")
print(f"Non-stationary series: {(~stationarity_df['is_stationary']).sum()} ({100*(~stationarity_df['is_stationary']).mean():.1f}%)")

# Save results
stationarity_df.to_csv('../data/interim/stationarity_results.csv', index=False)
print("\n✅ Saved: data/interim/stationarity_results.csv")

TESTING STATIONARITY

This may take a few minutes...



Testing stationarity: 100%|██████████| 1236/1236 [00:01<00:00, 740.04it/s]


✅ Stationarity testing complete!

Total series tested: 1187
Stationary series: 118 (9.9%)
Non-stationary series: 1069 (90.1%)

✅ Saved: data/interim/stationarity_results.csv


---
## 6️⃣ TRAIN/TEST SPLIT

**Time-based split** (NO random shuffle for time series!)

In [12]:
# Split at 2015 (80% train, 20% test approximately)
train_cutoff = 2015

train_data = df_features[df_features['year'] <= train_cutoff].copy()
test_data = df_features[df_features['year'] > train_cutoff].copy()

print("=" * 70)
print("TRAIN/TEST SPLIT")
print("=" * 70)
print(f"\nTrain: {train_data['year'].min()} - {train_data['year'].max()} ({len(train_data):,} records)")
print(f"Test:  {test_data['year'].min()} - {test_data['year'].max()} ({len(test_data):,} records)")
print(f"\nTrain %: {100 * len(train_data) / len(df_features):.1f}%")
print(f"Test %:  {100 * len(test_data) / len(df_features):.1f}%")

TRAIN/TEST SPLIT

Train: 1970 - 2015 (53,391 records)
Test:  2016 - 2021 (6,510 records)

Train %: 89.1%
Test %:  10.9%


---
## 7️⃣ SAVE PROCESSED DATA

In [13]:
print("=" * 70)
print("SAVING PROCESSED DATA")
print("=" * 70)

# Save full processed dataset
df_features.to_csv('../data/processed/emissions_processed.csv', index=False)
print("✅ Saved: data/processed/emissions_processed.csv")

# Save train/test splits
train_data.to_csv('../data/processed/train_data.csv', index=False)
print("✅ Saved: data/processed/train_data.csv")

test_data.to_csv('../data/processed/test_data.csv', index=False)
print("✅ Saved: data/processed/test_data.csv")

# Save metadata
metadata = {
    'total_records': len(df_features),
    'train_records': len(train_data),
    'test_records': len(test_data),
    'train_years': f"{train_data['year'].min()}-{train_data['year'].max()}",
    'test_years': f"{test_data['year'].min()}-{test_data['year'].max()}",
    'features': list(df_features.columns),
    'n_outliers_removed': int(df_clean['n_outliers_removed'].sum()),
    'n_stationary_series': int(stationarity_df['is_stationary'].sum()),
    'n_nonstationary_series': int((~stationarity_df['is_stationary']).sum())
}

with open('../data/processed/preprocessing_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
print("✅ Saved: data/processed/preprocessing_metadata.json")

print("\n" + "=" * 70)
print("📦 ALL DATA SAVED!")
print("=" * 70)

SAVING PROCESSED DATA
✅ Saved: data/processed/emissions_processed.csv
✅ Saved: data/processed/train_data.csv
✅ Saved: data/processed/test_data.csv
✅ Saved: data/processed/preprocessing_metadata.json

📦 ALL DATA SAVED!


---
## ✅ PREPROCESSING COMPLETE - SUMMARY

In [14]:
print("=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)

print("\n✅ COMPLETED STEPS:")
print("   1. Loaded raw data")
print("   2. Handled missing values")
print("   3. TIME-SERIES SPECIFIC outlier detection")
print(f"      - Outliers removed: {int(df_clean['n_outliers_removed'].sum())}")
print("   4. Feature engineering (lag, rolling stats, time features)")
print("   5. Stationarity testing")
print(f"      - Stationary: {int(stationarity_df['is_stationary'].sum())} series")
print(f"      - Non-stationary: {int((~stationarity_df['is_stationary']).sum())} series")
print("   6. Train/test split (time-based)")
print("   7. Saved all processed data")

print("\n📁 FILES CREATED:")
print("   • data/processed/emissions_processed.csv - Full dataset with features")
print("   • data/processed/train_data.csv - Training data (1970-2015)")
print("   • data/processed/test_data.csv - Test data (2016-2021)")
print("   • data/processed/preprocessing_metadata.json - Summary statistics")
print("   • data/interim/stationarity_results.csv - Stationarity test results")
print("   • reports/outlier_analysis/outlier_report.csv - Outlier detection report")

print("\n🎯 KEY STATISTICS:")
print(f"   Total records: {len(df_features):,}")
print(f"   Features created: {len([col for col in df_features.columns if col not in df.columns])}")
print(f"   Train records: {len(train_data):,} (1970-2015)")
print(f"   Test records: {len(test_data):,} (2016-2021)")

print("\n📝 NEXT STEPS:")
print("   1. Run 03_arima_model.ipynb for ARIMA training")
print("   2. Run 04_prophet_model.ipynb for Prophet training")
print("   3. Run 05_lstm_model.ipynb for LSTM training")
print("   4. Run 06_model_comparison.ipynb to compare all models")

print("\n" + "=" * 70)
print("✨ PREPROCESSING NOTEBOOK COMPLETE!")
print("=" * 70)

PREPROCESSING SUMMARY

✅ COMPLETED STEPS:
   1. Loaded raw data
   2. Handled missing values
   3. TIME-SERIES SPECIFIC outlier detection
      - Outliers removed: 5711
   4. Feature engineering (lag, rolling stats, time features)
   5. Stationarity testing
      - Stationary: 118 series
      - Non-stationary: 1069 series
   6. Train/test split (time-based)
   7. Saved all processed data

📁 FILES CREATED:
   • data/processed/emissions_processed.csv - Full dataset with features
   • data/processed/train_data.csv - Training data (1970-2015)
   • data/processed/test_data.csv - Test data (2016-2021)
   • data/processed/preprocessing_metadata.json - Summary statistics
   • data/interim/stationarity_results.csv - Stationarity test results
   • reports/outlier_analysis/outlier_report.csv - Outlier detection report

🎯 KEY STATISTICS:
   Total records: 59,901
   Features created: 10
   Train records: 53,391 (1970-2015)
   Test records: 6,510 (2016-2021)

📝 NEXT STEPS:
   1. Run 03_arima_model.